In [1]:
import json
import os
from sentence_transformers import SentenceTransformer
import chromadb

# ==========================================
# 1. ĐỌC DỮ LIỆU ĐÃ BĂM (CHUNKING)
# ==========================================
print("1. Đang đọc file JSON...")
with open('CHUNKED_RECURSIVE_DATA.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Tùy thuộc vào code của bạn Long, data có thể là list các chuỗi, hoặc list các dictionary.
# Đoạn code này sẽ trích xuất phần text ra thành một list chuẩn:
chunks = []
for item in data:
    if isinstance(item, str):
        chunks.append(item)
    elif isinstance(item, dict):
        # Thông thường LangChain lưu text vào key 'page_content' hoặc 'text'
        text_content = item.get('page_content', item.get('text', ''))
        if text_content:
            chunks.append(text_content)

print(f"-> Đã tìm thấy {len(chunks)} đoạn văn bản (chunks).")

# ==========================================
# 2. XỬ LÝ TIỀN TỐ CHO MODEL E5
# ==========================================
# Đây là "tử huyệt" của model E5: Bắt buộc phải thêm chữ "passage: " trước tài liệu
formatted_chunks = ["passage: " + chunk for chunk in chunks]

# ==========================================
# 3. TẢI MÔ HÌNH EMBEDDING
# ==========================================
print("\n2. Đang tải mô hình multilingual-e5-base...")
# Lưu ý: Lần đầu chạy nó sẽ tải khoảng 1.1GB model về máy, các lần sau sẽ rất nhanh
model = SentenceTransformer('intfloat/multilingual-e5-base')

# ==========================================
# 4. KHỞI TẠO VECTOR DATABASE
# ==========================================
print("\n3. Đang khởi tạo ChromaDB...")
# Dòng này sẽ tạo một thư mục tên 'db_khoa_cntt' ngay trong project của bạn để lưu file
chroma_client = chromadb.PersistentClient(path="./db_khoa_cntt")

# Xóa collection cũ nếu chạy lại nhiều lần để tránh trùng lặp
try:
    chroma_client.delete_collection("tai_lieu_khoa_cntt")
except:
    pass

collection = chroma_client.create_collection(name="tai_lieu_khoa_cntt")

# ==========================================
# 5. ÉP VECTOR VÀ LƯU VÀO KHO (BATCH PROCESSING)
# ==========================================
print("\n4. Đang ép Vector và lưu vào DB (Quá trình này tốn chút thời gian tùy RAM/GPU)...")

# Chia nhỏ ra ép từng đợt (batch) 32 đoạn một lúc để không bị tràn RAM máy tính
batch_size = 32 
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    batch_formatted = formatted_chunks[i:i+batch_size]
    
    # Ép text thành mảng số
    embeddings = model.encode(batch_formatted).tolist()
    
    # Tạo ID duy nhất cho mỗi đoạn
    ids = [f"chunk_{j}" for j in range(i, i + len(batch_chunks))]
    
    # Lưu vào ChromaDB
    collection.add(
        documents=batch_chunks,  # Lưu text gốc (KHÔNG có chữ passage:) để mốt lôi ra cho LLM đọc
        embeddings=embeddings,   # Lưu mảng số để so sánh
        ids=ids
    )
    print(f" -> Đã xử lý và lưu: {min(i + batch_size, len(chunks))} / {len(chunks)} chunks.")

print("\n🎉 HOÀN TẤT! Toàn bộ Vector đã được lưu an toàn tại thư mục ./db_khoa_cntt")

1. Đang đọc file JSON...
-> Đã tìm thấy 421 đoạn văn bản (chunks).

2. Đang tải mô hình multilingual-e5-base...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\TrieuNguyen\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TrieuNguyen\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


3. Đang khởi tạo ChromaDB...

4. Đang ép Vector và lưu vào DB (Quá trình này tốn chút thời gian tùy RAM/GPU)...
 -> Đã xử lý và lưu: 32 / 421 chunks.
 -> Đã xử lý và lưu: 64 / 421 chunks.
 -> Đã xử lý và lưu: 96 / 421 chunks.
 -> Đã xử lý và lưu: 128 / 421 chunks.
 -> Đã xử lý và lưu: 160 / 421 chunks.
 -> Đã xử lý và lưu: 192 / 421 chunks.
 -> Đã xử lý và lưu: 224 / 421 chunks.
 -> Đã xử lý và lưu: 256 / 421 chunks.
 -> Đã xử lý và lưu: 288 / 421 chunks.
 -> Đã xử lý và lưu: 320 / 421 chunks.
 -> Đã xử lý và lưu: 352 / 421 chunks.
 -> Đã xử lý và lưu: 384 / 421 chunks.
 -> Đã xử lý và lưu: 416 / 421 chunks.
 -> Đã xử lý và lưu: 421 / 421 chunks.

🎉 HOÀN TẤT! Toàn bộ Vector đã được lưu an toàn tại thư mục ./db_khoa_cntt
